# 06 - 使用 DeepSeek 分析一个文本块

这一节只分析 `AAPL_business_001`，目的是先验证 Prompt、结构化输出和原文引用，再考虑批量处理全部 31 个文本块。

```text
一个 chunk -> Prompt -> DeepSeek -> 结构化分析 -> Python 引用校验
```

> 运行模型调用单元格会产生一次 DeepSeek API 请求和少量费用。

## 1. 读取并选择一个 chunk

这里按 `chunk_id` 精确选择文本块，避免误调用全部数据。

In [ ]:
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data/sec/AAPL"
CHUNK_ID = "AAPL_business_001"

chunk_files = sorted(DATA_DIR.glob("*_chunks.json"), reverse=True)
if not chunk_files:
    raise FileNotFoundError("没有找到 chunks JSON，请先运行 05_chunk_sections.ipynb")

chunks_path = chunk_files[0]
chunks_data = json.loads(chunks_path.read_text(encoding="utf-8"))
selected_chunk = next(
    (chunk for chunk in chunks_data["chunks"] if chunk["chunk_id"] == CHUNK_ID),
    None,
)
if selected_chunk is None:
    raise ValueError(f"没有找到文本块：{CHUNK_ID}")

print(f"已选择：{selected_chunk['chunk_id']}")
print(f"章节：{selected_chunk['section_title']}")
print(f"字符数：{selected_chunk['char_count']:,}")
print(f"原文位置：{selected_chunk['source_start']:,} - {selected_chunk['source_end']:,}")

## 2. 定义模型必须返回的数据结构

普通聊天可能每次返回不同格式。Pydantic 模型规定了固定字段，便于后续批量保存、验证和汇总。`evidence_quote` 必须是英文原文中的逐字引用。

In [ ]:
class Finding(BaseModel):
    topic: str = Field(description="简短主题，例如产品、商业模式、市场或依赖关系")
    statement_cn: str = Field(description="只基于原文得出的简洁中文事实")
    evidence_quote: str = Field(
        description="支持该事实的英文原文逐字引用，不得翻译、省略或改写"
    )
    significance_cn: str = Field(description="这项事实对理解公司业务的重要性")


class ChunkAnalysis(BaseModel):
    chunk_id: str = Field(description="输入中提供的 chunk_id，必须原样返回")
    summary_cn: str = Field(description="本块内容的简洁中文摘要")
    findings: list[Finding] = Field(description="3 到 6 条有原文支持的重要事实")
    limitations_cn: list[str] = Field(
        description="仅根据当前文本块无法确定、需要其他章节补充的信息"
    )

## 3. 构造分析 Prompt

System Prompt 规定分析原则，Human Prompt 传入当前块的元数据和原文。明确禁止使用外部知识，可以降低模型把其他年份信息混进结果的风险。

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """你是一名审慎的公司财报研究助手。
只能依据用户提供的 SEC 财报原文分析，不使用外部知识，不提供投资建议。
所有结论使用简洁中文。每条事实都必须包含一段英文原文逐字引用。
引用不得翻译、改写、使用省略号或拼接不连续句子。
如果当前文本不足以支持某项判断，把它写入 limitations_cn，不要猜测。
必须只返回一个 JSON 对象，并包含以下字段：
chunk_id：输入的 chunk_id 字符串。
summary_cn：中文摘要字符串。
findings：对象数组，每个对象必须包含 topic、statement_cn、evidence_quote、significance_cn。
limitations_cn：当前文本无法确定的信息组成的字符串数组。""",
    ),
    (
        "human",
        """请分析下面这个财报文本块。

chunk_id: {chunk_id}
ticker: {ticker}
form: {form}
section: {section_title}
source_start: {source_start}
source_end: {source_end}

<filing_text>
{filing_text}
</filing_text>""",
    ),
])

print(prompt.invoke({
    "chunk_id": selected_chunk["chunk_id"],
    "ticker": selected_chunk["ticker"],
    "form": selected_chunk["form"],
    "section_title": selected_chunk["section_title"],
    "source_start": selected_chunk["source_start"],
    "source_end": selected_chunk["source_end"],
    "filing_text": selected_chunk["text"],
}).messages[0].content)

## 4. 创建 DeepSeek 模型和结构化分析链

`with_structured_output(ChunkAnalysis)` 会把返回格式约束成上面定义的 Pydantic 结构。`prompt | structured_llm` 表示 Prompt 的输出交给模型处理。

In [ ]:
load_dotenv(PROJECT_ROOT / ".env", override=True)

api_key = os.getenv("DEEPSEEK_API_KEY", "").strip()
if not api_key:
    raise ValueError("请先在 .env 中配置 DEEPSEEK_API_KEY")

model_name = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")

llm = ChatDeepSeek(
    model=model_name,
    api_key=api_key,
    base_url=base_url,
    temperature=0,
    request_timeout=60,
    max_retries=2,
)
structured_llm = llm.with_structured_output(
    ChunkAnalysis,
    method="json_mode",
)
analysis_chain = prompt | structured_llm

print(f"模型：{model_name}")
print("结构化分析链已创建")

## 5. 调用一次 DeepSeek

下面这个单元格会产生一次 API 请求。它只传入当前选中的一个 chunk。

In [ ]:
analysis = analysis_chain.invoke({
    "chunk_id": selected_chunk["chunk_id"],
    "ticker": selected_chunk["ticker"],
    "form": selected_chunk["form"],
    "section_title": selected_chunk["section_title"],
    "source_start": selected_chunk["source_start"],
    "source_end": selected_chunk["source_end"],
    "filing_text": selected_chunk["text"],
})

analysis

## 6. 用 Python 检查模型引用

模型声称引用原文并不等于引用一定正确。这里检查 `chunk_id` 是否一致，并在归一化 HTML 产生的空白和商标符号间距后，确认每段 `evidence_quote` 的文字仍能完整对应输入文本。

In [ ]:
def normalize_evidence(text: str) -> str:
    normalized = re.sub(r"\s+", " ", text).strip()
    normalized = re.sub(r"\s+([®™])", r"\1", normalized)
    normalized = re.sub(r"([®™])\s+([,.;:])", r"\1\2", normalized)
    return normalized


validation_errors = []
normalized_chunk_text = normalize_evidence(selected_chunk["text"])

if analysis.chunk_id != selected_chunk["chunk_id"]:
    validation_errors.append(
        f"chunk_id 不一致：{analysis.chunk_id}"
    )

for index, finding in enumerate(analysis.findings, start=1):
    quote = finding.evidence_quote.strip()
    if not quote:
        validation_errors.append(f"第 {index} 条事实没有原文引用")
    elif normalize_evidence(quote) not in normalized_chunk_text:
        validation_errors.append(f"第 {index} 条引用无法在原文中逐字找到")

if validation_errors:
    print("引用检查未通过：")
    for error in validation_errors:
        print(f"- {error}")
else:
    print(f"引用检查通过：{len(analysis.findings)} 条事实均有原文依据")

## 7. 以可读格式显示分析结果

In [ ]:
print(f"摘要：{analysis.summary_cn}\n")

for index, finding in enumerate(analysis.findings, start=1):
    print(f"[{index}] {finding.topic}")
    print(f"事实：{finding.statement_cn}")
    print(f"意义：{finding.significance_cn}")
    print(f"引用：{finding.evidence_quote}\n")

if analysis.limitations_cn:
    print("当前文本的局限：")
    for limitation in analysis.limitations_cn:
        print(f"- {limitation}")

## 8. 保存单块分析结果

保存时同时记录引用校验状态。批量处理时，可以跳过已经存在且校验通过的结果，从而支持断点续跑。

In [ ]:
output_data = {
    "source_chunks_file": str(chunks_path),
    "model": model_name,
    "chunk_metadata": {
        key: selected_chunk[key]
        for key in (
            "chunk_id",
            "ticker",
            "form",
            "section",
            "source_start",
            "source_end",
        )
    },
    "analysis": analysis.model_dump(),
    "citation_validation": {
        "passed": not validation_errors,
        "errors": validation_errors,
    },
}

output_path = DATA_DIR / f"{selected_chunk['chunk_id']}_analysis.json"
output_path.write_text(
    json.dumps(output_data, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"分析结果已保存：{output_path.resolve()}")

## 这一节完成了什么

现在我们完成了 Map 阶段的最小验证：一个文本块可以经过固定 Prompt 得到结构化事实，并由 Python 检查引用真实性。

下一步不是简单写一个 `for` 循环，而是增加失败重试、结果缓存、调用间隔和断点续跑，再安全地分析剩余文本块。